# Sugidanon — Code-Switch Hiligaynon ASR Benchmark

Hiligaynon (Ilonggo): over 9 million speakers, nearly invisible to modern
speech technology. This notebook reproduces, on a fresh cloud machine in a
few minutes, the finding that off-the-shelf models hear the English and
Tagalog in Ilonggo speech but miss the Hiligaynon itself.

It downloads the open dataset, runs a Whisper baseline, and scores the
switch penalty — word error rate at the moment the language switches.

Dataset: https://huggingface.co/datasets/LauelKills/sugidanon-hil-codeswitch
Code: https://github.com/Jazztinn/tinig-sa-liwanag
Site: https://tinig-sa-liwanag.vercel.app

Runtime -> Run all. No setup; about 3-5 minutes on the free CPU runtime.


## 1. Install


In [ ]:
!pip install -q openai-whisper huggingface_hub datasets
!apt-get -qq install -y ffmpeg >/dev/null


## 2. Get the code and dataset


In [ ]:
!git clone -q https://github.com/Jazztinn/tinig-sa-liwanag.git
%cd tinig-sa-liwanag
from huggingface_hub import snapshot_download
DS = snapshot_download('LauelKills/sugidanon-hil-codeswitch', repo_type='dataset')
print('dataset at:', DS)


## 3. Run the Whisper baseline
Whisper has no native Hiligaynon; Tagalog (tl) is the closest, which is the
point. Watch it transcribe the English and Tagalog words well and mangle the
Hiligaynon.


In [ ]:
!python scripts/run_whisper.py --audio-dir $DS/data/audio --out-dir /content/preds --model small --language tl


## 4. Score: overall / switch-region / monolingual WER + switch penalty


In [ ]:
!python score.py --ref $DS/data/annotations --hyp /content/preds


## What you just saw

A negative switch penalty: words next to a language switch scored better than
the monolingual Hiligaynon. The switch regions carry the borrowed English and
Tagalog words the model already knows; the Hiligaynon matrix is what it cannot
handle. tl-en is near-solved (about 6 percent error); hil-en is the worst
(about 41 percent). The failure scales with how much Hiligaynon is in the
sentence — exactly what this dataset measures.

Swap --model small for large-v3, or run Meta MMS, to put another model on the
same benchmark.


## Optional: developer one-liner
Audiofolder layout, so the dataset loads in a single line. If this cell errors,
ignore it — the benchmark above is independent.


In [ ]:
from datasets import load_dataset
ds = load_dataset('LauelKills/sugidanon-hil-codeswitch', data_dir='data/audio', split='train')
print(ds)
print(ds[0]['transcript'], '|', ds[0]['switch_type'])
